In [3]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Dense, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import categorical_crossentropy
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model
from tensorflow.keras.applications import imagenet_utils
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix
import itertools
import os
import shutil
from sklearn import metrics
import random
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
train_path = "C:\\Users\\Dell\\Desktop\\Dataset\\Train"
valid_path = "C:\\Users\\Dell\\Desktop\\Dataset\\Validation"
test_path = "C:\\Users\\Dell\\Desktop\\Dataset\\Test"

train_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.inception_v3.preprocess_input).flow_from_directory(
    directory=train_path, classes = ['BabyCry', 'Bark', 'Cough', 'Dishes', 'Doorbell', 'FireAlarm', 'SmokeDetector', 'Sneeze', 'Thunder', 'Water'], target_size=(224,224), batch_size=32)
valid_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.inception_v3.preprocess_input).flow_from_directory(
    directory=valid_path, classes = ['BabyCry', 'Bark', 'Cough', 'Dishes', 'Doorbell', 'FireAlarm', 'SmokeDetector', 'Sneeze', 'Thunder', 'Water'], target_size=(224,224), batch_size=32)
test_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.inception_v3.preprocess_input).flow_from_directory(
    directory=test_path, classes = ['BabyCry', 'Bark', 'Cough', 'Dishes', 'Doorbell', 'FireAlarm', 'SmokeDetector', 'Sneeze', 'Thunder', 'Water'], target_size=(224,224), batch_size=32, shuffle = False)

Found 4890 images belonging to 10 classes.
Found 524 images belonging to 10 classes.
Found 524 images belonging to 10 classes.


In [5]:
inc = tf.keras.applications.inception_v3.InceptionV3()
inc.summary()

Model: "inception_v3"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 299, 299, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv2d (Conv2D)                (None, 149, 149, 32  864         ['input_1[0][0]']                
                                )                                                                 
                                                                                                  
 batch_normalization (BatchNorm  (None, 149, 149, 32  96         ['conv2d[0][0]']                 
 alization)                     )                                                      

                                                                  'activation_7[0][0]',           
                                                                  'activation_10[0][0]',          
                                                                  'activation_11[0][0]']          
                                                                                                  
 conv2d_15 (Conv2D)             (None, 35, 35, 64)   16384       ['mixed0[0][0]']                 
                                                                                                  
 batch_normalization_15 (BatchN  (None, 35, 35, 64)  192         ['conv2d_15[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 activation_15 (Activation)     (None, 35, 35, 64)   0           ['batch_normalization_15[0][0]'] 
          

 conv2d_53 (Conv2D)             (None, 17, 17, 192)  215040      ['activation_52[0][0]']          
                                                                                                  
 conv2d_58 (Conv2D)             (None, 17, 17, 192)  215040      ['activation_57[0][0]']          
                                                                                                  
 conv2d_59 (Conv2D)             (None, 17, 17, 192)  147456      ['average_pooling2d_5[0][0]']    
                                                                                                  
 batch_normalization_50 (BatchN  (None, 17, 17, 192)  576        ['conv2d_50[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 batch_normalization_53 (BatchN  (None, 17, 17, 192)  576        ['conv2d_53[0][0]']              
 ormalizat

In [6]:
for layer in inc.layers[:-2]:
    layer.trainable = False
x = inc.layers[-2].output
#x = Flatten()(x)
#x = Dense(1024, activation = 'relu')(x)
output = Dense(units=10, activation='softmax')(x)
model = Model(inputs=inc.input, outputs=output)

model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 299, 299, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv2d (Conv2D)                (None, 149, 149, 32  864         ['input_1[0][0]']                
                                )                                                                 
                                                                                                  
 batch_normalization (BatchNorm  (None, 149, 149, 32  96         ['conv2d[0][0]']                 
 alization)                     )                                                             

 conv2d_27 (Conv2D)             (None, 35, 35, 64)   18432       ['mixed2[0][0]']                 
                                                                                                  
 batch_normalization_27 (BatchN  (None, 35, 35, 64)  192         ['conv2d_27[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 activation_27 (Activation)     (None, 35, 35, 64)   0           ['batch_normalization_27[0][0]'] 
                                                                                                  
 conv2d_28 (Conv2D)             (None, 35, 35, 96)   55296       ['activation_27[0][0]']          
                                                                                                  
 batch_normalization_28 (BatchN  (None, 35, 35, 96)  288         ['conv2d_28[0][0]']              
 ormalizat

 batch_normalization_40 (BatchN  (None, 17, 17, 192)  576        ['conv2d_40[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 batch_normalization_43 (BatchN  (None, 17, 17, 192)  576        ['conv2d_43[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 batch_normalization_48 (BatchN  (None, 17, 17, 192)  576        ['conv2d_48[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 batch_normalization_49 (BatchN  (None, 17, 17, 192)  576        ['conv2d_49[0][0]']              
 ormalizat

Total params: 21,823,274
Trainable params: 20,490
Non-trainable params: 21,802,784
__________________________________________________________________________________________________


In [7]:
model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(x=train_batches,
            steps_per_epoch=len(train_batches),
            validation_data=valid_batches,
            validation_steps=len(valid_batches),
            epochs=15,
)


Epoch 1/15
153/153 [==============================] - 653s 4s/step - loss: 1.4573 - accuracy: 0.5029 - val_loss: 1.5980 - val_accuracy: 0.4618
Epoch 2/15
153/153 [==============================] - 552s 4s/step - loss: 1.0630 - accuracy: 0.6454 - val_loss: 1.4945 - val_accuracy: 0.4924
Epoch 3/15
153/153 [==============================] - 530s 3s/step - loss: 0.9129 - accuracy: 0.7041 - val_loss: 1.5919 - val_accuracy: 0.4523
Epoch 4/15
153/153 [==============================] - 565s 4s/step - loss: 0.8367 - accuracy: 0.7331 - val_loss: 1.4104 - val_accuracy: 0.5057
Epoch 5/15
153/153 [==============================] - 559s 4s/step - loss: 0.7585 - accuracy: 0.7589 - val_loss: 1.3190 - val_accuracy: 0.5573
Epoch 6/15
153/153 [==============================] - 550s 4s/step - loss: 0.7135 - accuracy: 0.7800 - val_loss: 1.3366 - val_accuracy: 0.5649
Epoch 7/15
153/153 [==============================] - 546s 4s/step - loss: 0.6645 - accuracy: 0.7908 - val_loss: 1.2990 - val_accuracy: 0.5515

In [8]:
predictions = model.predict(x=test_batches, verbose=1)
from sklearn import metrics
metrics.accuracy_score(test_batches.classes, np.argmax(predictions, axis = -1))

17/17 [==============================] - 47s 2s/step


0.5954198473282443

In [9]:
import os.path
model.save("D:\\SavedModels\\INCV3Latest.h5")

In [10]:
np.argmax(predictions, axis = -1)

array([5, 6, 0, 0, 9, 0, 1, 0, 9, 0, 0, 1, 0, 0, 0, 3, 9, 7, 0, 0, 0, 5,
       0, 5, 5, 1, 1, 3, 5, 9, 2, 3, 0, 9, 3, 0, 0, 7, 2, 0, 8, 0, 8, 0,
       3, 0, 0, 0, 0, 2, 0, 8, 0, 0, 0, 0, 0, 0, 0, 8, 2, 2, 9, 8, 9, 1,
       1, 1, 7, 1, 9, 0, 3, 1, 1, 1, 1, 2, 1, 3, 1, 1, 1, 7, 1, 3, 3, 1,
       3, 1, 1, 1, 1, 1, 1, 2, 2, 1, 8, 1, 2, 9, 1, 1, 3, 7, 1, 0, 1, 0,
       5, 3, 1, 1, 1, 1, 2, 9, 1, 7, 2, 7, 2, 8, 1, 2, 1, 0, 2, 9, 3, 2,
       1, 3, 2, 2, 3, 1, 2, 1, 1, 2, 9, 3, 7, 9, 2, 2, 2, 1, 1, 1, 6, 2,
       2, 8, 0, 2, 1, 2, 2, 4, 2, 2, 9, 2, 3, 3, 6, 9, 1, 1, 5, 7, 7, 5,
       3, 2, 7, 2, 5, 8, 3, 3, 5, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 6, 3, 7, 3, 3, 3, 3, 3, 9, 3, 3, 5, 7, 3, 3, 3, 3, 3, 3, 3,
       9, 3, 3, 3, 3, 3, 3, 6, 9, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 4,
       3, 6, 4, 4, 4, 4, 4, 4, 4, 3, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4,
       4, 0, 6, 3, 4, 1, 4, 3, 7, 6, 6, 9, 6, 6, 5, 5, 4, 5, 5, 7, 4, 8,
       5, 5, 5, 6, 5, 5, 6, 5, 5, 5, 5, 6, 5, 5, 5,

In [11]:
test_batches.classes

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
       5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,

In [ ]:
|